# Vault Enterprise on Minikube

The complete demo is started from the repository root with **`./demo.sh`**. This notebook documents only the Vault stage and uses the same scripts, without deleting Minikube profiles or reinitializing an existing cluster.

[Installation and input secrets](README.md) · [Workspace review](docs/WORKSPACE_REVIEW.md).

Prepare `.env` from `.env.example`: license, external integration credentials and image configuration. Do not add `VAULT_TOKEN`, `VAULT_ADDR` or `VAULT_CACERT`: the launcher generates the bootstrap after initializing Vault. The generated state (CA, initialization keys, administrative token, logs and PID) is stored in `~/.local/state/lumen-vault-demo`, outside `/tmp` and the workspace. `LUMEN_STATE_DIR` allows a different location. Adopting an already initialized Vault requires its saved keys; they are not regenerated.


In [ ]:
%%bash
set -euo pipefail
./demo.sh check


## Rerunnable deployment

Creates or starts the `workshop` profile; existing resources are preserved. On an empty installation it pulls the `josemerchan/vault-xai:2.2.0-ent` image, which bundles the xAI plugin (AMD64/ARM64), from Docker Hub, deploys three Raft nodes with TLS, initializes once and unseals the nodes. `.env` lets you set `VAULT_IMAGE_REPOSITORY` and `VAULT_IMAGE_TAG`. Port 8443 listens on `127.0.0.1`.

If Vault already exists, it checks the image and adopts the keys from the previous lab; it does not replace the image or regenerate certificates. The plugin catalog is global, but the xAI engine used by Lumen is configured in `verify`.


In [ ]:
%%bash
set -euo pipefail
bash scripts/demo/vault.sh


In [ ]:
%%bash
source scripts/demo/common.sh
load_inputs
load_vault
vault status
kubectl --context workshop -n vault get pods
helm --kube-context workshop -n vault list


## `verify` namespace, integration secrets and OAuth profile

The following notebooks work in the `verify` namespace. This cell creates it if missing, imports the `.env` inputs into `verify/secrets` without displaying them, registers the RAR type `vault:path_access` in Verify and configures the Vault OAuth profile. It can be rerun and preserves user passwords that are already stored.


In [ ]:
%%bash
set -euo pipefail
bash scripts/demo/bootstrap.sh
bash verify/scripts/configure-base.sh


To resume, use `./demo.sh start`; do not run `minikube delete --all`, delete keys manually or run `operator init` again. `./demo.sh stop` stops only local processes and keeps the cluster and data. To tear down the whole lab, review `./demo.sh remove --dry-run` and run `./demo.sh remove`: it deletes the three namespaces/PVCs and the Lumen resources in Verify, keeps `.env` and allows a new `start`. [Scope of the removal](README.md#volver-a-empezar-desde-cero). Single-share initialization is a lab simplification.
